# Signal Quality Diagnostics

Run comprehensive signal quality diagnostics on neural recordings: channel quality
detection, SNR analysis, power spectral density, trial quality checks, and
channel correlation analysis. Generate a full quality report at the end.

## 1. Imports

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from src.config import get_default_config
from src.data.loader import load_willett_dataset
from src.diagnostics.channel_quality import detect_bad_channels
from src.diagnostics.snr_analysis import compute_snr
from src.diagnostics.spectral_analysis import compute_psd
from src.diagnostics.trial_quality import detect_bad_trials
from src.diagnostics.correlation_analysis import compute_channel_correlation
from src.diagnostics.report_generator import generate_quality_report

## 2. Load Data

In [ ]:
cfg = get_default_config()
trial_index = load_willett_dataset(cfg)

# Load first trial as a representative sample
sample_row = trial_index.iloc[0]
signal = np.load(sample_row['signal_path'])
print(f"Loaded trial {sample_row['trial_id']} with shape {signal.shape}")

## 3. Channel Quality Detection

Identify bad channels based on variance, amplitude range, and flatline detection.

In [ ]:
bad_channels = detect_bad_channels(signal)
print(f"Bad channels detected: {len(bad_channels)}")
if len(bad_channels) > 0:
    print(f"Bad channel indices: {bad_channels}")

## 4. Channel Variance

Plot per-channel variance to identify noisy or dead channels.

In [ ]:
channel_var = np.var(signal, axis=0)

fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(range(len(channel_var)), channel_var, color='steelblue', alpha=0.8)
if len(bad_channels) > 0:
    ax.bar(bad_channels, channel_var[bad_channels], color='red', alpha=0.8, label='Bad channels')
    ax.legend()
ax.set_xlabel('Channel Index')
ax.set_ylabel('Variance')
ax.set_title('Per-Channel Variance')
plt.tight_layout()
plt.show()

## 5. Compute SNR

Estimate signal-to-noise ratio across channels.

In [ ]:
snr_values = compute_snr(signal)
print(f"Mean SNR: {np.mean(snr_values):.2f} dB")
print(f"Min  SNR: {np.min(snr_values):.2f} dB")
print(f"Max  SNR: {np.max(snr_values):.2f} dB")

## 6. SNR Distribution

Plot the distribution of SNR values across all channels.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(snr_values, bins=30, color='teal', edgecolor='white', alpha=0.8)
ax.axvline(np.mean(snr_values), color='red', linestyle='--', label=f'Mean={np.mean(snr_values):.1f} dB')
ax.set_xlabel('SNR (dB)')
ax.set_ylabel('Count')
ax.set_title('Channel SNR Distribution')
ax.legend()
plt.tight_layout()
plt.show()

## 7. Power Spectral Density

Compute and visualize PSD using Welch's method.

In [ ]:
freqs, psd = compute_psd(signal)

fig, ax = plt.subplots(figsize=(10, 4))
# Plot mean PSD across channels with shaded std region
mean_psd = np.mean(psd, axis=0) if psd.ndim > 1 else psd
ax.semilogy(freqs, mean_psd, color='navy')
if psd.ndim > 1:
    std_psd = np.std(psd, axis=0)
    ax.fill_between(freqs, mean_psd - std_psd, mean_psd + std_psd, alpha=0.2, color='navy')
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('Power Spectral Density')
ax.set_title('Mean PSD Across Channels (Welch)')
plt.tight_layout()
plt.show()

## 8. Trial Quality Check

Run quality checks across multiple trials to identify bad recordings.

In [ ]:
# Load a batch of trials for quality assessment
signals = []
for i in range(min(20, len(trial_index))):
    row = trial_index.iloc[i]
    sig = np.load(row['signal_path'])
    signals.append(sig)

bad_trials = detect_bad_trials(signals)
print(f"Trials checked: {len(signals)}")
print(f"Bad trials: {len(bad_trials)}")
if len(bad_trials) > 0:
    print(f"Bad trial indices: {bad_trials}")

## 9. Correlation Analysis

Compute pairwise channel correlation to detect redundant or highly correlated channels.

In [ ]:
corr_matrix = compute_channel_correlation(signal)

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(corr_matrix, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
plt.colorbar(im, ax=ax, label='Correlation')
ax.set_xlabel('Channel')
ax.set_ylabel('Channel')
ax.set_title('Channel-Channel Correlation Matrix')
plt.tight_layout()
plt.show()

# Print summary statistics
upper_tri = corr_matrix[np.triu_indices_from(corr_matrix, k=1)]
print(f"Mean pairwise correlation: {np.mean(upper_tri):.4f}")
print(f"Max  pairwise correlation: {np.max(upper_tri):.4f}")
print(f"Highly correlated pairs (>0.9): {np.sum(upper_tri > 0.9)}")

## 10. Generate Full Quality Report

Compile all diagnostics into a single quality report.

In [ ]:
report = generate_quality_report(signal)
print(report)

---

**Next:** Proceed to `03_preprocessing_qc.ipynb` for preprocessing quality control.